# Alzheimer's Detection – Model Training & Selection

Trains **Custom CNN**, **MobileNetV2**, and **EfficientNetB0**; saves the best by test macro F1.

- **Kaggle**: dataset `/kaggle/input/alzheimers-detection-dataset/dataset`, output `/kaggle/working/models/`
- **Local**: dataset `./dataset`, output `./models`

## Run all in one cell (imports, config, data, models, training, save best)

In [ ]:
# --- Imports ---
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# --- Config ---
IS_KAGGLE = os.path.isdir("/kaggle/input")
DATASET_DIR = "/kaggle/input/alzheimers-detection-dataset/dataset" if IS_KAGGLE else os.path.join(os.getcwd(), "dataset")
MODELS_DIR = "/kaggle/working/models" if IS_KAGGLE else os.path.join(os.getcwd(), "models")
SAVED_MODEL_PATH = os.path.join(MODELS_DIR, "best_alzheimer_model.keras")
CLASS_NAMES_PATH = os.path.join(MODELS_DIR, "class_names.txt")
CLASS_NAMES = ["Mild_Demented", "Moderate_Demented", "Non_Demented", "Very_Mild_Demented"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = (224, 224)
IMG_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT, TEST_SPLIT = 0.2, 0.1
EPOCHS, PATIENCE = 55, 12
OVERSAMPLE_MIN_PER_CLASS, CLASS_WEIGHT_MAX = 1200, 10.0
LABEL_SMOOTHING, FINETUNE_EPOCHS, FINETUNE_LR = 0.1, 15, 1e-5
UNFREEZE_FRACTION = 0.25
IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".gif")
tf.random.set_seed(SEED)
np.random.seed(SEED)
os.makedirs(MODELS_DIR, exist_ok=True)
print("TensorFlow:", tf.__version__, "| Kaggle" if IS_KAGGLE else "| Local", "| Dataset:", DATASET_DIR)

# --- Data: collect, split, oversample, class weights ---
def collect_paths_labels():
    paths, labels = [], []
    for i, cname in enumerate(CLASS_NAMES):
        d = os.path.join(DATASET_DIR, cname)
        if not os.path.isdir(d): continue
        for f in os.listdir(d):
            if f.lower().endswith(IMG_EXTENSIONS):
                paths.append(os.path.join(d, f))
                labels.append(i)
    return np.array(paths), np.array(labels)

paths, labels = collect_paths_labels()
if len(paths) == 0: raise FileNotFoundError("No images in " + str(DATASET_DIR))
X_tv, X_test, y_tv, y_test = train_test_split(paths, labels, test_size=TEST_SPLIT, stratify=labels, random_state=SEED)
val_ratio = VAL_SPLIT / (1 - TEST_SPLIT)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=val_ratio, stratify=y_tv, random_state=SEED)

def oversample(X, y, target=OVERSAMPLE_MIN_PER_CLASS):
    Xl, yl = [], []
    for c in range(NUM_CLASSES):
        m = y == c
        Xc, nc = X[m], m.sum()
        if nc == 0: continue
        idx = np.tile(np.arange(nc), (target + nc - 1) // nc)[:max(nc, target)]
        np.random.RandomState(SEED).shuffle(idx)
        Xl.append(Xc[idx]); yl.append(np.full(len(idx), c))
    out_x = np.concatenate(Xl); out_y = np.concatenate(yl)
    p = np.random.RandomState(SEED).permutation(len(out_x))
    return out_x[p], out_y[p]

X_train, y_train = oversample(X_train, y_train)
class_weights = {i: min(float(w), CLASS_WEIGHT_MAX) for i, w in enumerate(compute_class_weight("balanced", classes=np.unique(y_train), y=y_train))}
print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test), "| Class weights:", class_weights)

# --- tf.data with augmentation ---
def load_preprocess(path, label, aug=False):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if aug:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.image.random_saturation(img, 0.9, 1.1)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
        img = tf.clip_by_value(img, 0, 1)
    return img, label

def make_ds(paths, labels, batch, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths.astype(str), labels.astype(np.int32)))
    if shuffle: ds = ds.shuffle(min(len(paths), 2048), seed=SEED)
    ds = ds.map(lambda p, l: load_preprocess(p, l, augment), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, BATCH_SIZE, True, True)
val_ds = make_ds(X_val, y_val, BATCH_SIZE, False)
test_ds = make_ds(X_test, y_test, BATCH_SIZE, False)

# --- Models ---
def build_custom_cnn():
    inp = keras.Input(shape=IMG_SHAPE)
    x = layers.Conv2D(96, 3, activation="relu", padding="same")(inp)
    x = layers.BatchNormalization()(x); x = layers.MaxPool2D(2)(x); x = layers.Dropout(0.2)(x)
    x = layers.Conv2D(192, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPool2D(2)(x); x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(384, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPool2D(2)(x); x = layers.Dropout(0.3)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    return keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="custom_cnn")

def build_mobilenetv2():
    base = MobileNetV2(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inp = keras.Input(shape=IMG_SHAPE)
    x = base(inp); x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)
    return keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="mobilenetv2")

def build_efficientnetb0():
    base = EfficientNetB0(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inp = keras.Input(shape=IMG_SHAPE)
    x = base(inp); x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)
    return keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="efficientnetb0")

BUILDERS = {"custom_cnn": build_custom_cnn, "mobilenetv2": build_mobilenetv2, "efficientnetb0": build_efficientnetb0}

# --- Train and evaluate (with fine-tuning for transfer models) ---
def train_eval(name, train_ds, val_ds, test_ds, y_test, cw):
    model = BUILDERS[name]()
    lr = 1e-4 if name == "efficientnetb0" else 1e-3
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [EarlyStopping("val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
          ReduceLROnPlateau("val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)]
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=cw, callbacks=cb, verbose=1)
    if name in ("mobilenetv2", "efficientnetb0"):
        base = model.layers[1]
        n, cut = len(base.layers), int((1 - UNFREEZE_FRACTION) * len(base.layers))
        for l in base.layers[:cut]: l.trainable = False
        for l in base.layers[cut:]: l.trainable = True
        model.compile(optimizer=keras.optimizers.Adam(FINETUNE_LR), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        print("Fine-tuning top 25% backbone...")
        model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, class_weight=cw, callbacks=cb, verbose=1)
    y_pred = np.argmax(model.predict(test_ds), axis=1)
    acc = np.mean(y_test == y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    return {"model": model, "test_accuracy": acc, "test_f1_macro": f1,
            "classification_report": classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0),
            "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()}

# --- Train all, pick best, save ---
results = {}
for name in BUILDERS:
    print("\n" + "="*60 + "\nTraining", name.upper() + "\n" + "="*60)
    try:
        results[name] = train_eval(name, train_ds, val_ds, test_ds, y_test, class_weights)
        print("Test accuracy:", round(results[name]["test_accuracy"], 4), "| Test macro F1:", round(results[name]["test_f1_macro"], 4))
    except Exception as e:
        print("Error:", e)
        results[name] = None

best_name = max((n for n, r in results.items() if r), key=lambda n: results[n]["test_f1_macro"], default=None)
if best_name is None: raise RuntimeError("No model trained.")
best_f1 = results[best_name]["test_f1_macro"]
print("\nBest model:", best_name, "(test macro F1:", round(best_f1, 4), ")")
results[best_name]["model"].save(SAVED_MODEL_PATH)
with open(CLASS_NAMES_PATH, "w") as f: f.write("\n".join(CLASS_NAMES))
with open(os.path.join(MODELS_DIR, "best_model_metrics.json"), "w") as f:
    json.dump({"best_model": best_name, "test_accuracy": results[best_name]["test_accuracy"],
               "test_f1_macro": best_f1, "classification_report": results[best_name]["classification_report"],
               "confusion_matrix": results[best_name]["confusion_matrix"]}, f, indent=2)
print("Model saved →", SAVED_MODEL_PATH)
print("\nClassification report:\n", results[best_name]["classification_report"])

## 1. Imports

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("TensorFlow version:", tf.__version__)
print("Running on Kaggle environment" if os.path.isdir("/kaggle/input") else "Running locally")

## 2. Config (Kaggle / local)

In [ ]:
# Paths
IS_KAGGLE = os.path.isdir("/kaggle/input")
if IS_KAGGLE:
    DATASET_DIR = "/kaggle/input/alzheimers-detection-dataset/dataset"
    MODELS_DIR = "/kaggle/working/models"
else:
    DATASET_DIR = os.path.join(os.getcwd(), "dataset")
    MODELS_DIR = os.path.join(os.getcwd(), "models")

SAVED_MODEL_PATH = os.path.join(MODELS_DIR, "best_alzheimer_model.keras")
CLASS_NAMES_PATH = os.path.join(MODELS_DIR, "class_names.txt")

# Training
CLASS_NAMES = ["Mild_Demented", "Moderate_Demented", "Non_Demented", "Very_Mild_Demented"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = (224, 224)
IMG_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1
EPOCHS = 55
PATIENCE = 12
OVERSAMPLE_MIN_PER_CLASS = 1200
CLASS_WEIGHT_MAX = 10.0
LABEL_SMOOTHING = 0.1
FINETUNE_EPOCHS = 15
FINETUNE_LR = 1e-5
UNFREEZE_FRACTION = 0.25
IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".gif")

tf.random.set_seed(SEED)
np.random.seed(SEED)

os.makedirs(MODELS_DIR, exist_ok=True)
print("Dataset:", DATASET_DIR)
print("Output:", MODELS_DIR)

## 3. Data loading, split, oversampling

In [ ]:
def collect_image_paths_and_labels():
    image_paths, labels = [], []
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(DATASET_DIR, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(IMG_EXTENSIONS):
                image_paths.append(os.path.join(class_dir, fname))
                labels.append(class_idx)
    return np.array(image_paths), np.array(labels)


def load_and_split_data():
    paths, labels = collect_image_paths_and_labels()
    if len(paths) == 0:
        raise FileNotFoundError("No images in " + str(DATASET_DIR) + " for " + str(CLASS_NAMES))
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        paths, labels, test_size=TEST_SPLIT, stratify=labels, random_state=SEED
    )
    val_ratio = VAL_SPLIT / (1 - TEST_SPLIT)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_ratio, stratify=y_trainval, random_state=SEED
    )
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)


def oversample_minority_classes(X_train, y_train, target_min=OVERSAMPLE_MIN_PER_CLASS):
    X_list, y_list = [], []
    for c in range(NUM_CLASSES):
        mask = y_train == c
        X_c, y_c = X_train[mask], y_train[mask]
        n = len(X_c)
        if n == 0:
            continue
        n_repeat = max(1, (target_min + n - 1) // n)
        indices = np.tile(np.arange(n), n_repeat)[:max(n, target_min)]
        np.random.RandomState(SEED).shuffle(indices)
        X_list.append(X_c[indices])
        y_list.append(np.full(len(indices), c))
    X_bal = np.concatenate(X_list, axis=0)
    y_bal = np.concatenate(y_list, axis=0)
    perm = np.random.RandomState(SEED).permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]


def get_class_weights(y, max_weight=CLASS_WEIGHT_MAX):
    weights = compute_class_weight("balanced", classes=np.unique(y), y=y)
    return {i: min(float(w), max_weight) for i, w in enumerate(weights)}


# Load and oversample
(X_train, y_train), (X_val, y_val), (X_test, y_test) = load_and_split_data()
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
X_train, y_train = oversample_minority_classes(X_train, y_train)
print(f"After oversampling - Train: {len(X_train)}")
class_weights = get_class_weights(y_train)
print("Class weights (capped):", class_weights)

## 4. Build tf.data datasets

In [ ]:
def _load_and_preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.image.random_saturation(img, 0.9, 1.1)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
        img = tf.clip_by_value(img, 0, 1)
    return img, label


def dataset_from_paths(paths, labels, batch_size, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths.astype(str), labels.astype(np.int32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 2048), seed=SEED)
    ds = ds.map(
        lambda p, l: _load_and_preprocess(p, l, augment=augment),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = dataset_from_paths(X_train, y_train, BATCH_SIZE, shuffle=True, augment=True)
val_ds = dataset_from_paths(X_val, y_val, BATCH_SIZE, shuffle=False)
test_ds = dataset_from_paths(X_test, y_test, BATCH_SIZE, shuffle=False)
print("Train/val/test datasets ready.")

## 5. Model builders

In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=IMG_SHAPE)
    x = layers.Conv2D(96, 3, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv2D(192, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D(2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(384, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D(2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")


def build_mobilenetv2():
    base = MobileNetV2(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inputs = keras.Input(shape=IMG_SHAPE)
    x = base(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2")


def build_efficientnetb0():
    base = EfficientNetB0(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inputs = keras.Input(shape=IMG_SHAPE)
    x = base(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="efficientnetb0")


MODEL_BUILDERS = {
    "custom_cnn": build_custom_cnn,
    "mobilenetv2": build_mobilenetv2,
    "efficientnetb0": build_efficientnetb0,
}

## 6. Train and evaluate one model

In [ ]:
def train_and_evaluate(model_name, train_ds, val_ds, test_ds, y_test, class_weights):
    model = MODEL_BUILDERS[model_name]()
    lr = 1e-4 if model_name == "efficientnetb0" else 1e-3
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        class_weight=class_weights, callbacks=callbacks, verbose=1,
    )
    if model_name in ("mobilenetv2", "efficientnetb0"):
        base = model.layers[1]
        n = len(base.layers)
        cut = int((1 - UNFREEZE_FRACTION) * n)
        for layer in base.layers[:cut]:
            layer.trainable = False
        for layer in base.layers[cut:]:
            layer.trainable = True
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=FINETUNE_LR),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        print("Fine-tuning top", UNFREEZE_FRACTION * 100, "% of backbone...")
        model.fit(
            train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS,
            class_weight=class_weights, callbacks=callbacks, verbose=1,
        )
    y_pred = np.argmax(model.predict(test_ds), axis=1)
    test_acc = np.mean(y_test == y_pred)
    test_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0)
    cm = confusion_matrix(y_test, y_pred).tolist()
    return {
        "model": model,
        "history": history.history,
        "test_accuracy": float(test_acc),
        "test_f1_macro": float(test_f1),
        "classification_report": report,
        "confusion_matrix": cm,
    }

print("Function train_and_evaluate defined.")

## 7. Train all models and select best

In [ ]:
results = {}
for name in MODEL_BUILDERS:
    print(f"\n{'='*60}\nTraining {name.upper()}\n{'='*60}")
    try:
        results[name] = train_and_evaluate(
            name, train_ds, val_ds, test_ds, y_test, class_weights
        )
        print(f"Test accuracy : {results[name]['test_accuracy']:.4f}")
        print(f"Test macro F1 : {results[name]['test_f1_macro']:.4f}")
    except Exception as e:
        print(f"Error: {e}")
        results[name] = None

## 8. Save best model and metrics

In [ ]:
best_name = None
best_f1 = -1
for name, res in results.items():
    if res and res["test_f1_macro"] > best_f1:
        best_f1 = res["test_f1_macro"]
        best_name = name

if best_name is None:
    raise RuntimeError("No model trained successfully.")

print(f"Best model: {best_name} (test macro F1: {best_f1:.4f})")
best_model = results[best_name]["model"]
best_model.save(SAVED_MODEL_PATH)
with open(CLASS_NAMES_PATH, "w") as f:
    f.write("\n".join(CLASS_NAMES))

metrics_path = os.path.join(MODELS_DIR, "best_model_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(
        {
            "best_model": best_name,
            "test_accuracy": results[best_name]["test_accuracy"],
            "test_f1_macro": results[best_name]["test_f1_macro"],
            "classification_report": results[best_name]["classification_report"],
            "confusion_matrix": results[best_name]["confusion_matrix"],
        },
        f,
        indent=2,
    )

print(f"Model saved → {SAVED_MODEL_PATH}")
print(f"Class names → {CLASS_NAMES_PATH}")
print(f"Metrics     → {metrics_path}")
print("\nClassification report:\n", results[best_name]["classification_report"])